In [96]:
# =========================
# 1. Install dependencies
# =========================
# pip install pandas numpy scikit-learn torch transformers openai tqdm

# =========================
# 2. Imports + config
# =========================
import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from openai import OpenAI

DATA_PATH = "datset.csv"   # change to "dataset.csv" if needed
OPENAI_MODEL = "gpt-5-mini"   # lower-cost LLM for feature engineering
DISTILBERT_MODEL = "distilbert-base-uncased"
MAX_LEN = 256
LOOKBACK_TXNS = 10
USE_LLM = True
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
import os


client = OpenAI()


In [97]:
DATA_PATH="../../transactions/card_transaction.v1.csv"
df = pd.read_csv(DATA_PATH)

In [125]:


# ================================
# 🔥 CLEAN DATA (MANDATORY)
# ================================

# 1. Clean Amount ($ → float)
df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

# 2. Convert Time → Hour (0–23)
df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

# If Time is numeric seconds instead:
# df["Hour"] = (df["Time"] // 3600) % 24

# 3. Convert Use Chip → 0/1
df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

# 4. Convert Errors → 0/1
df["Errors?"] = (df["Errors?"] != "None").astype(int)

# 5. Convert Zip → numeric
df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")
# Try numeric first
df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

# Convert seconds → hour
df["Hour"] = (df["Time"] // 3600) % 24
df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})
# 6. Fill missing values
df = df.fillna(0)

# ================================
# 🔍 VERIFY (VERY IMPORTANT)
# ================================
print(df.dtypes)
print(df.head())

/tmp/ipykernel_3335815/728849075.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour


User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time              float64
Amount            float64
Use Chip          float64
Merchant Name       int64
Merchant City      object
Merchant State     object
Zip               float64
MCC                 int64
Errors?             int64
Is Fraud?           int64
Hour              float64
dtype: object
   User  Card  Year  Month  Day  Time  Amount  Use Chip        Merchant Name  \
0     0     0  2002      9    1   0.0  134.09       0.0  3527213246127876953   
1     0     0  2002      9    1   0.0   38.48       0.0  -727612092139916043   
2     0     0  2002      9    2   0.0  120.34       0.0  -727612092139916043   
3     0     0  2002      9    2   0.0  128.95       0.0  3414527459579106770   
4     0     0  2002      9    3   0.0  104.71       0.0  5817218446178736267   

   Merchant City Merchant State      Zip   MCC  Errors?  Is Fraud?  Hour  
0       La Ver

In [126]:
df["Merchant City"] = df["Merchant City"].astype(str)
df["Merchant State"] = df["Merchant State"].astype(str)

In [127]:
print(df.dtypes)
print(df.head())

User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time              float64
Amount            float64
Use Chip          float64
Merchant Name       int64
Merchant City      object
Merchant State     object
Zip               float64
MCC                 int64
Errors?             int64
Is Fraud?           int64
Hour              float64
dtype: object
   User  Card  Year  Month  Day  Time  Amount  Use Chip        Merchant Name  \
0     0     0  2002      9    1   0.0  134.09       0.0  3527213246127876953   
1     0     0  2002      9    1   0.0   38.48       0.0  -727612092139916043   
2     0     0  2002      9    2   0.0  120.34       0.0  -727612092139916043   
3     0     0  2002      9    2   0.0  128.95       0.0  3414527459579106770   
4     0     0  2002      9    3   0.0  104.71       0.0  5817218446178736267   

   Merchant City Merchant State      Zip   MCC  Errors?  Is Fraud?  Hour  
0       La Ver

In [98]:
# 🔥 Model

from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

In [100]:
# for m in genai.list_models():
#     print(m.name, "->", m.supported_generation_methods)

In [101]:
model = genai.GenerativeModel("models/gemini-flash-lite-latest")


In [102]:
# ================================
# 🔥 FORCE CLEAN RAW DATA
# ================================

# Amount → remove $ and convert
df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

# Time → convert to hour
df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

# If Time is numeric already (seconds), use:
# df["Hour"] = (df["Time"] // 3600) % 24

# Use Chip → 0/1
df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

# Errors → 0/1
df["Errors?"] = (df["Errors?"] != "None").astype(int)

# Zip → numeric
df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

# Fill missing
df = df.fillna(0)

/tmp/ipykernel_3335815/2132959710.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour


In [128]:
PROMPT = """
You are a fraud analytics feature engineering assistant.

The features you generate will be used to train a RandomForestClassifier.

Dataset columns and TYPES (STRICT — DO NOT ASSUME ANYTHING ELSE):

- User → int64 (group key)
- Card → int64 (DO NOT USE)
- Year → int64
- Month → int64
- Day → int64
- Time → float64 (transaction timestamp in seconds — DO NOT USE directly)
- Hour → float64 (0–23, derived from Time — USE THIS)
- Amount → float64 (numeric)
- Use Chip → float64 (1 = Yes, 0 = No)
- Merchant Name → int64 (encoded — DO NOT USE directly)
- Merchant City → object (categorical — ONLY use nunique/count)
- Merchant State → object (categorical — ONLY use nunique/count)
- Zip → float64 (numeric)
- MCC → int64 (category code — can use nunique/count)
- Errors? → int64 (1 = error, 0 = no error)
- Is Fraud? → int64 (target label — DO NOT USE)

YOUR TASK:

Design GLOBAL USER-LEVEL FEATURES.

STRICT REQUIREMENTS (VERY IMPORTANT):

- Features must be NUMERIC only (int or float)
- Must be directly usable in sklearn (no preprocessing needed)
- Must use pandas groupby('User')
- MUST be computed ONLY from df (no dependency on other generated features)
- SAME features for ALL users

ALLOWED OPERATIONS:

- mean, std, min, max, sum
- count, nunique
- ratios (division of aggregates)

AVOID:

- raw text usage
- using Merchant Name directly
- using Is Fraud?
- string operations
- referencing other generated features

IMPORTANT RULES:

- Always use "Hour" for time-based features (NOT Time)
- Merchant City / State → ONLY nunique or count
- Ensure all outputs are numeric scalars per user

OUTPUT FORMAT (STRICT JSON):

[
  {
    "feature_name": "...",
    "pandas_code": "...",
    "description": "...",
    "type": "numeric"
  }
]

Generate 12–15 high-quality fraud detection features.

Focus on:
- spending patterns
- transaction frequency
- merchant/category diversity
- geographic spread
- time behavior
- error behavior
- chip usage patterns

Do NOT compute values.
Do NOT output anything outside JSON.
"""

In [131]:
model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")

In [132]:
response = model_llm.generate_content(PROMPT)

output = response.text
# print("Raw Output:\n", output)

output = output.replace("```json", "").replace("```", "").strip()

features = json.loads(output)

# =========================================
# ✅ APPLY FEATURES USING PANDAS
# =========================================

# Example: load your dataset
# df = pd.read_csv("your_dataset.csv")

user_df = pd.DataFrame()

for f in features:
    try:
        print("Applying:", f["feature_name"])
        user_df[f["feature_name"]] = eval(f["pandas_code"])
    except Exception as e:
        print("Error in", f["feature_name"], ":", e)

# ✅ Add label
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

# ✅ Fill missing
user_df = user_df.fillna(0)

print("\nFinal Shape:", user_df.shape)
print(user_df.head())

Applying: user_tx_count
Applying: user_avg_amount
Applying: user_std_amount
Applying: user_tx_hour_mean
Applying: user_tx_hour_std
Applying: user_chip_usage_ratio
Applying: user_error_rate
Applying: user_merchant_city_nunique
Applying: user_merchant_state_nunique
Applying: user_mcc_nunique
Applying: user_zip_change_std
Applying: user_city_tx_ratio
Applying: user_avg_amount_per_city_ratio
Applying: user_merchant_city_count

Final Shape: (2000, 15)
      user_tx_count  user_avg_amount  user_std_amount  user_tx_hour_mean  \
User                                                                       
0             19963        81.299989        94.159093                0.0   
1              8919        81.118050       156.784691                0.0   
2             41978        35.159687        54.298552                0.0   
3             10117       117.277603       268.654472                0.0   
4             18542        97.011698       138.619564                0.0   

      user_tx_ho

In [133]:
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

In [ ]:
# Separate label first
y = user_df["is_fraud_user"]

# Remove non-numeric features
X = user_df.drop(columns=["is_fraud_user"])
X = X.select_dtypes(include=["number"])

In [135]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # IMPORTANT for fraud datasets
)

In [136]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"   # IMPORTANT for fraud
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [137]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.88      0.69      0.78       131
           1       0.87      0.96      0.91       269

    accuracy                           0.87       400
   macro avg       0.87      0.83      0.84       400
weighted avg       0.87      0.87      0.87       400

ROC-AUC: 0.9049348732937939


In [138]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance.head(10))

user_tx_count                     0.188518
user_city_tx_ratio                0.171675
user_merchant_city_count          0.158882
user_mcc_nunique                  0.115221
user_merchant_city_nunique        0.107084
user_merchant_state_nunique       0.100098
user_zip_change_std               0.044285
user_std_amount                   0.043194
user_avg_amount                   0.036960
user_avg_amount_per_city_ratio    0.034083
dtype: float64


In [139]:
feature_names = [f["feature_name"] for f in features]

In [140]:
top_features = importance.head(10).to_dict()

In [141]:
report1 = classification_report(y_test, y_pred)
roc1 = roc_auc_score(y_test, y_prob)

llm_input = {
    "existing_features": feature_names,
    "classification_report": report1,
    "roc_auc": roc1,
    "top_features": top_features
}

In [155]:
PROMPT = f"""
You are a fraud analytics feature engineering expert.

A RandomForestClassifier was trained on USER-LEVEL features.

==============================
DATASET SCHEMA (STRICT)
==============================

- User → int64 (group key)
- Card → int64 (DO NOT USE)
- Year → int64
- Month → int64
- Day → int64
- Time → float64 (timestamp in seconds — DO NOT USE directly)
- Hour → float64 (0–23 — USE THIS for time features)
- Amount → float64
- Use Chip → float64 (1 = Yes, 0 = No)
- Merchant Name → int64 (DO NOT USE directly)
- Merchant City → object (ONLY use nunique/count)
- Merchant State → object (ONLY use nunique/count)
- Zip → float64
- MCC → int64
- Errors? → int64 (1 = error, 0 = no error)
- Is Fraud? → int64 (TARGET — DO NOT USE)

==============================
CURRENT FEATURES
==============================
{llm_input["existing_features"]}

==============================
MODEL PERFORMANCE
==============================

Classification Report:
{llm_input["classification_report"]}

ROC-AUC:
{llm_input["roc_auc"]}

Top Important Features:
{llm_input["top_features"]}

==============================
YOUR TASK
==============================

1. Analyze weaknesses in the model (especially fraud recall)
2. Identify missing behavioral signals
3. Suggest 8–12 NEW or IMPROVED features

==============================
STRICT RULES (VERY IMPORTANT)
==============================

- Features must be NUMERIC only
- Must work directly in sklearn (no preprocessing)
- Must use pandas groupby('User')
- MUST be directly computable from df
- MUST NOT depend on other generated features
- MUST NOT duplicate existing features

ALLOWED:
- mean, std, min, max, sum
- count, nunique
- ratios

AVOID:
- raw text usage
- using Merchant Name directly
- using Is Fraud?
- referencing other generated features
- using Time (use Hour only)

IMPORTANT:
- Each pandas_code MUST return a pandas Series indexed by User

==============================
OUTPUT FORMAT (STRICT JSON)
==============================

[
  {{
    "feature_name": "...",
    "pandas_code": "...",
    "reason": "..."
  }}
]

Focus on:
- behavioral anomalies
- spending variability
- temporal patterns (Hour-based)
- transaction bursts
- merchant/category diversity
- geographic spread
- error behavior
- chip usage changes

Do NOT compute values.
Do NOT output anything outside JSON.
"""

In [156]:
model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")


In [157]:
response = model_llm.generate_content(PROMPT)

output = response.text.replace("```json", "").replace("```", "").strip()

new_features = json.loads(output)

In [158]:
base_features = []

for f in features:
    code = f["pandas_code"]
    
    # Only keep df-based features
    if "df.groupby" in code and "user_" not in code.split("=")[-1]:
        base_features.append(f)

In [159]:
# # Amount → float
# df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

# # Time → hour (0–23)
# df["Hour"] = pd.to_datetime(df["Time"], format="%H:%M:%S", errors="coerce").dt.hour

# # Use Chip → 0/1
# df["Use Chip"] = (df["Use Chip"] == "Yes").astype(int)

# # Errors → 0/1
# df["Errors?"] = (df["Errors?"] != "None").astype(int)

# # Zip → numeric
# df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

# # Fill missing
# df = df.fillna(0)

In [160]:
for f in base_features:
    try:
        user_df[f["feature_name"]] = eval(f["pandas_code"])
    except Exception as e:
        print("Base error:", f["feature_name"], e)

In [161]:
X = user_df.drop(columns=["is_fraud_user"])
y = user_df["is_fraud_user"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [162]:
# ================================
# 🔹 9. TRAIN MODEL
# ================================
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# ================================
# 🔹 10. PREDICT
# ================================
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# ================================
# 🔹 11. EVALUATE
# ================================
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("\n=== ROC AUC ===")
print(roc_auc_score(y_test, y_prob))

# ================================
# 🔹 12. FEATURE IMPORTANCE
# ================================
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\n=== Top Features ===")
print(importance.head(10))


=== Classification Report ===
              precision    recall  f1-score   support

           0       0.90      0.69      0.78       131
           1       0.87      0.96      0.91       269

    accuracy                           0.88       400
   macro avg       0.88      0.83      0.85       400
weighted avg       0.88      0.88      0.87       400


=== ROC AUC ===
0.9027214166122761

=== Top Features ===
user_tx_count                     0.177505
user_city_tx_ratio                0.164542
user_merchant_city_count          0.155953
user_mcc_nunique                  0.116284
user_merchant_city_nunique        0.108388
user_merchant_state_nunique       0.102826
user_std_amount                   0.048543
user_zip_change_std               0.046680
user_avg_amount                   0.040045
user_avg_amount_per_city_ratio    0.039234
dtype: float64


In [163]:
report2 = classification_report(y_test, y_pred)
roc2 = roc_auc_score(y_test, y_prob)

In [164]:
# precision    recall  f1-score   support

#           No       0.87      0.72      0.79       131
#          Yes       0.87      0.95      0.91       269

#     accuracy                           0.87       400
#    macro avg       0.87      0.83      0.85       400
# weighted avg       0.87      0.87      0.87       400

# ROC-AUC: 0.9174210391895343

In [167]:
print(report1)
print(report2)


              precision    recall  f1-score   support

           0       0.88      0.69      0.78       131
           1       0.87      0.96      0.91       269

    accuracy                           0.87       400
   macro avg       0.87      0.83      0.84       400
weighted avg       0.87      0.87      0.87       400

              precision    recall  f1-score   support

           0       0.90      0.69      0.78       131
           1       0.87      0.96      0.91       269

    accuracy                           0.88       400
   macro avg       0.88      0.83      0.85       400
weighted avg       0.88      0.88      0.87       400



In [166]:
print(roc1)
print(roc2)

0.9049348732937939
0.9027214166122761
